In [9]:
import sympy as sym
import math 
import numpy as np
from sympy import totient
import plotly.offline as pyo
import plotly.graph_objs as go
import sympy as sym
from scipy.stats import qmc  
import plotly.subplots as sp
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [1]:
# 
# LowDiscrepancySequence
# Modify_Sobol
class LowDiscrepancySequence :
    def __init__(self, dim: int, n0: int, m: int):
        """
        Args:
            dim (int): The dimensionality of the sequence.
            n0 (int): The initial index of the sequence.
            n (int): The number of points to generate.
        """
        self.n0 = n0
        self.npts = 2**m
        self.dim = dim
        self.b = 2
        self.Y = self.get_Y()
    

    def get_seq(self, name):
        
        if name == 'vdc':
            len_seq = int(self.Y .max())
            seq = self.van_der_corput(len_seq)
            return self.seq_pts(seq)
            
        elif name == 'vdc_gc':
            len_seq = int(self.Y .max())
            seq = self.van_der_corput_gray(len_seq)
            return self.seq_pts(seq)
            
        elif name == 'sobol_farey':
            len_seq = int(self.Y .max())
            seq = self.farey_seq(len_seq)            
            return self.seq_pts(seq)
            
        elif name == 'sobol_kronecker':
            len_seq = int(self.Y .max())
            seq = self.kronecker_1D(len_seq)            
            return self.seq_pts(seq)

        elif name == 'kronecker':            
            return self.kronecker(self.npts)

        elif name == 'faure':            
            return self.faure(self.npts)

        elif name == 'halton':            
            return self.halton(self.npts)
            
        elif name == 'sobol_lcg':
            len_seq = int(self.Y .max())
            seq = self.lcg(len_seq)            
            return self.seq_pts(seq)
            
        else:
            raise ValueError(f"Unknown sequence name: {name}")

    def seq_pts(self, seq):
        P = np.zeros((self.npts, self.dim))
        for dim_i in range(self.dim):
            for n_i in range(self.npts):
                y_i = int(self.Y [n_i][dim_i] - 1)
                P[n_i][dim_i] = seq[y_i]
        return P
 

    
    # def inv_seq_pts(self, seq, P):
   
    #     Y = np.zeros((self.npts, self.dim), dtype=int)
        
    #     for n_i in range(self.npts):
    #         for dim_i in range(self.dim):
    #             p_val = P[n_i, dim_i]
    #             # Находим индекс в seq, для которого seq[index] == p_val.
    #             idx = np.where(seq == p_val)[0]
    #             if idx.size == 0:
    #                 raise ValueError(f"Значение {p_val} из P не найдено в seq")
    #             # Выбираем первый найденный индекс и прибавляем 1, чтобы получить Y.
    #             Y[n_i, dim_i] = idx[0] + 1
    #     return Y

###########################################################################################################################
    def b_ary(self, k, b):
        """
        Converts an integer k to a base-b representation.

        Args:
            k (int): The integer to convert (k >= 0).

        Returns:
            np.ndarray: A numpy array representing the base-b number.
        """
        if k == 0:
            return np.array([0])

        a = []
        j_max = int(np.log(k) / np.log(b))
        a = np.zeros(j_max + 1, dtype=int)
        q = b ** j_max
        for j in range(0, j_max + 1):
            a[j] = int(k / q)
            k = k - q * a[j]
            q = q // b
        return a

    def nextb_ary(self, a_in, b):
        """
        Computes the next number in the base-b representation.

        Args:
            a_in (np.ndarray): Current base-b number.

        Returns:
            np.ndarray: The next base-b number.
        """
        m = len(a_in)
        carry = True
        a_out = np.zeros(m, dtype=int)

        for i in range(m - 1, -1, -1):
            if carry:
                if a_in[i] == b - 1:
                    a_out[i] = 0
                else:
                    a_out[i] = a_in[i] + 1
                    carry = False
            else:
                a_out[i] = a_in[i]

        if carry:
            a_out = np.insert(a_out, 0, 1)

        return a_out

    def find_c_vec(self, d: int) -> int:
        """
        Reads a file and finds the s and a values for a given d, then converts them to a number.

        Args:
            d (int): The index of the row in the file.

        Returns:
            int: The converted value based on s and a.
        """
        dir_file = 'C:/Users/Vlada/Desktop/Sirius/NIR/code/data_sobol.21201'
        with open(dir_file, 'r') as infile:
            lines = infile.readlines()
            line = lines[d - 1].split()
            s, a = map(int, line[1:3])
            p = self.num_base_2(s, a)
        return int(p)

    def num_base_2(self, s: int, a: int) -> int:
        """
        Converts the values s and a to a binary number.

        Args:
            s (int): Degree of the polynomial.
            a (int): The integer to convert to binary.

        Returns:
            int: The decimal value of the modified binary representation.
        """
        if s == 0 and a == 0:
            num_a_2 = 1
        else:
            num_a = ''
            while a > 0:
                ost = a % 2
                num_a += str(ost)
                a //= 2

            delta = s - len(str(num_a)) - 1
            num_a = '1' + '0' * delta + num_a[::-1] + '1'
            num_a_2 = self.binary_to_decimal(num_a)
        return num_a_2

    def binary_to_decimal(self, binary_str: str) -> int:
        """
        Converts a binary string to a decimal integer.

        Args:
            binary_str (str): The binary string to convert.

        Returns:
            int: The decimal value of the binary string.
        """
        decimal_num = 0
        power = 0
        for digit in binary_str[::-1]:
            if digit == '1':
                decimal_num += 2 ** power
            power += 1
        return decimal_num

    def read_m(self, d: int) -> list[int]:
        """
        Reads a file and returns the initial m_i values for a given d.

        Args:
            d (int): The index of the row in the file.

        Returns:
            list[int]: A list of integers representing the m_i values.
        """
        if d == 1:
            m = [1]
        else:
            dir_file = 'C:/Users/Vlada/Desktop/Sirius/NIR/code/Sobol/new-joe-kuo-6.21201'
            with open(dir_file, 'r') as infile:
                lines = infile.readlines()
                line = lines[d - 1].split()
                m = list(map(int, line[3:]))
        return m

    def generate_MJ(self, C_vec: list[int], minit: list[int], r: int) -> list[int]:
        """
        Generates a sequence based on the C_vec coefficients and minit initial sequence.

        Args:
            C_vec (list[int]): A list of coefficients for the linear recurrence relation.
            minit (list[int]): The initial sequence.
            r (int): The desired length of the sequence.

        Returns:
            list[int]: The generated sequence.
        """
        q = len(C_vec) - 1
        sequence = minit[:]

        for j in range(len(minit), r):
            mj = 0
            for i in range(q):
                if C_vec[i]:
                    a = ((2 ** (i + 1)) * C_vec[i] * sequence[j - (i + 1)])
                    mj ^= a

            mj ^= (sequence[j - q])
            sequence.append(mj)

        return sequence

    def sobolmat(self, cvec: list[int], minit: list[int], r: int):
        """
        Generates a matrix based on the given coefficients and initial sequence.

        Args:
            cvec (list[int]): The coefficients for the recurrence relation.
            minit (list[int]): The initial sequence.
            r (int): The size of the resulting matrix.

        Returns:
            np.ndarray: The generated matrix.
        """
        q = len(cvec) - 1
        if q == 0:
            V = np.eye(r, dtype=int)
        else:
            mvec = list(minit) + [0] * (r - len(minit))

            for i in range(q, r):
                m_next = 0
                for l in range(q):
                    if cvec[l]:
                        m_next ^= ((2 ** (l + 1)) * cvec[l] * mvec[i - (l + 1)])

                m_next ^= mvec[i - q]
                mvec[i] = m_next

            V = np.zeros((r, r), dtype=int)
            for j in range(r):
                m_bin = self.b_ary(mvec[j],2)
                k = len(m_bin)
                for i in range(k):
                    V[j - i, j] = m_bin[k - i - 1]

        return V

    
    ############################################################################
    def get_Y(self):

        pvec = [1] + [self.find_c_vec(d) for d in range(2, self.dim + 1)]
        mmat =  [[0]] + [self.read_m(d) for d in range(2, self.dim + 1)]
        
        nmax = self.n0 + self.npts - 1
        rmax = 1 + int(np.log2(nmax))
        Y_list = np.zeros((self.npts, self.dim))
        y = np.zeros(rmax, dtype=int)

        r = (1 + int(np.log2(self.n0 - 1))) if (self.n0-1) > 1 else 1
        a = self.b_ary(self.n0 - 1,2)
        qnext = 2 ** r

        V_list = []

        for i in range(self.dim):
            q = int(np.log2(pvec[i]))
            cvec = self.b_ary(pvec[i],2)
            mj = mmat[i][:q]
            V = self.sobolmat(cvec, mj, rmax)
            V_list.append(V)

        bpwrs = np.array([(2 ** i) for i in range(0, rmax)])

        for k in range(self.npts):
            a = self.nextb_ary(a,2)
            if k + self.n0 == qnext:
                r += 1
                qnext *= 2

            for j in range(r):
                Y_list[k, 0] += bpwrs[j] * a[r - j - 1]

            for i in range(1, self.dim):
                V = V_list[i]
                for m in range(r):
                    for n in range(r):
                        y[m] += V[m, n] * a[r - n - 1]
                    y[m] %= 2
                    Y_list[k, i] += bpwrs[m] * y[m]
                    y[m] = 0

        return Y_list
        
################################## # farey_ # ################################## 

    def temp_farey_seq_classic(self, N):
    
        a, b, c, d = 0, 1, 1, N
        seq = [a / b]  
        
        while c <= N:
            k = (N + b) // d
            a, b, c, d = c, d, k * c - a, k * d - b
            seq.append(a / b)
        
        return np.array(seq, dtype=float)

    
    def farey_seq(self, seq_len):
        S = np.zeros(seq_len)
        cnt = 0
        N_cnt = 1
        
        while cnt != seq_len:
            farey = self.temp_farey_seq_classic(N_cnt)
            N_cnt +=1
            
            for num in farey:
                if cnt == seq_len:
                    break
                if num not in S:
                    S[cnt] = num  
                    cnt += 1
        return S
  
################################## # van_der_corput # ################################## 
    
    def van_der_corput(self, len_seq):
        b = 2
        S = np.zeros(len_seq)
        
        for k in range(len_seq):
            a = self.b_ary(self.n0 + k, 2)
            S[k] = np.dot(a[::-1], 1 / (b ** np.arange(1, len(a) + 1)))
        
        return S
    
################################## # van_der_corput_gray # ################################## 
    
    def van_der_corput_gray(self, len_seq):
        b = 2
        def gray_code(n: int) -> int:
            return n ^ (n >> 1)

        S = np.zeros(len_seq)
        
        for k in range(len_seq):
            gray_k = gray_code(self.n0 + k)
            a = self.b_ary(gray_k, 2)
            S[k] = np.dot(a[::-1], 1 / (b ** np.arange(1, len(a) + 1)))
        
        return S

################################## # Kroneker # ################################## 
 
    
    def generate_primes(self, n0, count): 
        primes = []
        n = n0
        while len(primes) < count:
            if sym.isprime(n):
                primes.append(n)
            n += 1 
        return primes
    
    def kronecker_1D(self, n):
        alpha = np.pi 
        arr = np.arange(1, n + 1) 
        seq = np.mod(arr * alpha, 1)
        return seq
        
    def kronecker(self, npts):
     
        sqrt_primes = [] # base == j and number(№) of dimention
        primes = self.generate_primes(self.n0, dim)
        for num in primes:
            sqrt_primes.append(np.sqrt(num))
        
        arr = np.array(range(self.n0, npts))    
    
        sec = [[] for _ in range(dim)]
        for num in arr:
            for j in range(len(sqrt_primes)):
                sec[j].append(round((num * sqrt_primes[j]) % 1, 10))
    
        sec = np.array(sec)
        return sec.T



################################## # Fore # ################################## 
    def fauremat(self, r, i):

        # создаём пустую матрицу r×r
        C = np.zeros((r, r), dtype=int)
        
        # базовые условия:
        C[0, 0] = 1
        for m in range(1, r):
            # диагональ
            C[m, m] = 1
            # первая строка
            C[0, m] = i * C[0, m - 1]
        
        for n in range(2, r):
            for m in range(1, n):
                C[m, n] = C[m - 1, n - 1] + i * C[m, n - 1]
        
        return C

    def find_base(self):
        prime_arr = list(sym.primerange(self.dim, self.dim + 100)) # простые числа
        base = prime_arr[0] # base - простое число равное или больше числу измерений
        return base

    def faure(self, npts):
        b = self.find_base() 
        
        nmax = self.n0 + npts - 1
        rmax = 1 + int(np.log2(nmax) / np.log2(b))

        P = np.zeros((npts, self.dim))
        y = np.zeros(rmax, dtype=int)

        r = (1 + int(np.log2(self.n0 - 1) / np.log2(b))) if (self.n0-1) > 1 else 1
        
        a = self.b_ary(self.n0 - 1, b)
        qnext = b ** r

        C_list = []

        for i in range(self.dim):#####################
            
            C = self.fauremat(rmax, i)
            C_list.append(C)

        # bpwrs = np.array([(b ** i) for i in range(0, rmax)])
        bpwrs = np.array([1 / (b ** i) for i in range(1, rmax + 1)])

    
        for k in range(npts):
            a = self.nextb_ary(a, b)
            if k + self.n0 == qnext:
                r += 1
                qnext = b * qnext

            for j in range(r):
                P[k, 0] += bpwrs[j] * a[r - j - 1]
            
            for i in range(1, self.dim):
                C = C_list[i]
                for m in range(r):
                    for n in range(r):
                        y[m] += C[m, n] * a[r - n - 1]
                    y[m] %= b
                    P[k, i] += bpwrs[m] * y[m]
                    y[m] = 0

        return P
################################## # halton # ################################## 

    
    def b_radical_ary(self, base, n):
        result = 0
        f = 1
        while n > 0:
            f = f / base
            result += f * (n % base)
            n = n // base
        return result
        
    def van_der_corput_element(self, base, n_i):
        return self.b_radical_ary(base, n_i)
    
    def van_der_corput(self, base, n0, n):
        sec = []
        for i in range(n0, n):
            sec.append(self.van_der_corput_element(base, i))
        return np.array(sec)
    
    # создание простого числа для массива базисов
    def is_prime(self, num):
        if num < 2:
            return False
        for i in range(2, int(num**0.5) + 1):
            if num % i == 0:
                return False
        return True
        
    # генерация простых чисел для базисов
    def generate_prime_numbers_array(self, N):
        prime_numbers = []
        num = 2
        while len(prime_numbers) < N:
            if self.is_prime(num):
                prime_numbers.append(num)
            num += 1
        return prime_numbers
        
    def halton(self, npts):
        sec = []    
        base = self.generate_prime_numbers_array(self.dim)
        for i in range(len(base)):  
            sec.append(self.van_der_corput(base[i], self.n0, npts))
        return np.array(sec).T
    


 ################################## # LCG  # ################################## 
   
    def lcg(self, n):
        """
        Generates n pseudorandom numbers in [0, 1) using a linear congruential generator (LCG).
        
        Parameters:
        - n (int): number of random values to generate
        - seed (int): initial seed X0 (default: 1)
        - a (int): multiplier (default: 16807, Park–Miller minimal standard)
        - c (int): increment (default: 0)
        - m (int): modulus (default: 2^31 - 1)
        
        Returns:
        - List[float]: pseudorandom numbers in [0, 1)
        """
        seed=1
        a=16807
        c=0
        m=2**31 - 1
        x = seed % m
        sequence = []
        for _ in range(n):
            x = (a * x + c) % m
            sequence.append(x / m)
        return sequence

    
    


In [18]:
# dim = 5
# m = 4
# n0 = 1
# # b = 3
# seq = Modify_Sobol( dim, n0, m)

# seq.get_seq('lcg')